# ReAct baseline (five SURF models)

One tool-calling loop per CSV. The file is registered as the same dataset context the agent uses, and the model may call the single-CSV tools. The CSV text is not placed in the prompt. After the loop, one structured call writes the same metadata schema as the single-LLM baseline.

PANGAEA uses `single_llm_pangaea_standard`. iNaturalist uses `croissant_inaturalist_standard`. A turn is sent only when the growing transcript plus a 4096-token completion budget fits the model's hardcoded context window.

Restart behavior:

- `{id}_react.json` means the output passed validation.
- `{id}_react.skipped.json` means the turn did not fit the context window, the call timed out, or the attempt failed.
- Each dataset is sent once. Successes, skips, and failed attempts are all kept on the next run.

Run the setup cell, then the smoke cell for 10 PANGAEA datasets. The last cell skips PANGAEA and resumes iNaturalist. After a kernel restart, rerun the setup cell before either run.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

BASE = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/config.py').is_file()),
    None,
)
if BASE is None:
    raise FileNotFoundError('Could not locate the metadata_agent project root')
sys.path.insert(0, str(BASE))
load_dotenv(BASE / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

from src.baseline.ReAct import CONTEXT_WINDOWS, DEFAULT_MODELS, PANGAEA_FULL_STANDARD_NAME, run_all_models

OUTPUTS_ROOT = BASE / 'outputs'
PROCESS_TIMEOUT_SECONDS = 600
INATURALIST_DIR_CANDIDATES = [
    BASE / 'data/experiment/inaturalist',
    BASE / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
]


def resolve_inaturalist_dir() -> Path:
    for path in INATURALIST_DIR_CANDIDATES:
        if path.is_dir() and any(path.glob('*.csv')):
            return path
    searched = '\n  '.join(str(path) for path in INATURALIST_DIR_CANDIDATES)
    raise FileNotFoundError(f'iNaturalist data directory not found. Tried:\n  {searched}')


EXPERIMENTS = [
    {
        'dataset_label': 'PANGAEA',
        'experiment': 'pangaea',
        'standard_name': PANGAEA_FULL_STANDARD_NAME,
        'data_dir': BASE / 'data/experiment/pangaea_datasets',
        'expected': 75,
    },
    {
        'dataset_label': 'iNaturalist',
        'experiment': 'inaturalist',
        'standard_name': 'croissant_inaturalist_standard',
        'data_dir': resolve_inaturalist_dir(),
        'expected': 100,
    },
]

if not os.getenv('SURF_API_KEY'):
    raise ValueError('SURF_API_KEY is missing from metadata_agent/.env')
if not os.getenv('SURF_API_BASE'):
    raise ValueError('SURF_API_BASE is missing from metadata_agent/.env')

print(f'Project: {BASE}')
print(f'Models: {len(DEFAULT_MODELS)}')
for model in DEFAULT_MODELS:
    print(f'  {model}: context window {CONTEXT_WINDOWS[model]}')
for experiment in EXPERIMENTS:
    csv_files = sorted(experiment['data_dir'].glob('*.csv'))
    print(
        f"{experiment['dataset_label']}: {len(csv_files)} files, "
        f"standard {experiment['standard_name']}"
    )
    if len(csv_files) != experiment['expected']:
        raise ValueError(
            f"Expected {experiment['expected']} {experiment['dataset_label']} CSV files, "
            f"found {len(csv_files)}"
        )

Project: /home/com3dian/Github/metadata_agent
Models: 5
  default-text-large: context window 131072
  Qwen/Qwen3.6-35B-A3B-FP8: context window 262144
  Sehyo/Qwen3.5-122B-A10B-NVFP4: context window 262144
  mistralai/Devstral-Small-2-24B-Instruct-2512: context window 262144
  mistralai/Mistral-Small-3.2-24B-Instruct-2506: context window 131072
PANGAEA: 75 files, standard single_llm_pangaea_standard
iNaturalist: 100 files, standard croissant_inaturalist_standard


## Smoke: 10 PANGAEA datasets, all five models

Runs the first 10 PANGAEA CSVs through every model. Results are written under `outputs/react_pangaea_smoke_<model>/`, separate from the full run. Restarting this cell skips datasets that already have a valid result.

In [2]:
# import json

# from src.baseline.ReAct.runner import model_slug, run_model

# SMOKE_N = 10
# smoke = EXPERIMENTS[0]
# smoke_files = sorted(smoke['data_dir'].glob('*.csv'))[:SMOKE_N]
# print(f"Smoke files: {len(smoke_files)}")
# for path in smoke_files:
#     print(f"  {path.name}")

# smoke_summaries = []
# for model in DEFAULT_MODELS:
#     smoke_summaries.append(
#         run_model(
#             model=model,
#             csv_files=smoke_files,
#             outputs_root=OUTPUTS_ROOT,
#             standard_name=smoke['standard_name'],
#             timeout_seconds=PROCESS_TIMEOUT_SECONDS,
#             experiment='pangaea_smoke',
#         )
#     )

# print('\n' + '=' * 78)
# for model in DEFAULT_MODELS:
#     output_dir = OUTPUTS_ROOT / f"react_pangaea_smoke_{model_slug(model)}"
#     print(f"\n{model}")
#     for path in smoke_files:
#         success = output_dir / f"{path.stem}_react.json"
#         skipped = output_dir / f"{path.stem}_react.skipped.json"
#         if success.is_file():
#             record = json.loads(success.read_text(encoding='utf-8'))
#             metadata = record.get('metadata') or {}
#             name = metadata.get('name') or metadata.get('description') or ''
#             tools = ', '.join(call.get('tool', '') for call in record.get('tool_calls') or [])
#             print(f"  OK  {path.name}: {name}")
#             print(f"      tools: {tools or 'none'}")
#         elif skipped.is_file():
#             record = json.loads(skipped.read_text(encoding='utf-8'))
#             print(f"  SKIP {path.name}: {record.get('reason')}")
#         else:
#             print(f"  FAIL {path.name}: no result file")
# smoke_summaries

## Resume iNaturalist, Qwen models only

PANGAEA is left as already recorded. This cell runs the two Qwen models on iNaturalist. A dataset that already has a result file is not sent again, and that file is not replaced.

In [3]:
inat = next(item for item in EXPERIMENTS if item['experiment'] == 'inaturalist')
qwen_models = [model for model in DEFAULT_MODELS if 'Qwen' in model]
print('\n' + '#' * 78)
print(inat['dataset_label'])
print('Qwen models:', ', '.join(qwen_models))
summaries = run_all_models(
    data_dir=inat['data_dir'],
    outputs_root=OUTPUTS_ROOT,
    standard_name=inat['standard_name'],
    models=qwen_models,
    timeout_seconds=PROCESS_TIMEOUT_SECONDS,
    experiment=inat['experiment'],
    dataset_label=inat['dataset_label'],
)
summaries


##############################################################################
iNaturalist
Qwen models: Qwen/Qwen3.6-35B-A3B-FP8, Sehyo/Qwen3.5-122B-A10B-NVFP4
iNaturalist datasets: 100
Standard: croissant_inaturalist_standard
Models: 2

MODEL 1/2

react model: Qwen/Qwen3.6-35B-A3B-FP8
Standard: croissant_inaturalist_standard
Output: /home/com3dian/Github/metadata_agent/outputs/react_inaturalist_Qwen_Qwen3.6-35B-A3B-FP8
Resuming with 100/100 datasets remaining; 0 existing result files kept.

[1/100] acianthus_fornicatus.csv


/home/com3dian/Github/metadata_agent/src/baseline/ReAct/runner.py:301: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  outcome = describe_csv(csv_path, **kwargs)


  OK in 6.972s tools: get_context_overview, detect_temporal_columns, get_temporal_extent, detect_spatial_columns, get_spatial_extent -> acianthus_fornicatus_react.json

[2/100] acontia_cretata.csv


/home/com3dian/Github/metadata_agent/src/baseline/ReAct/runner.py:301: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  outcome = describe_csv(csv_path, **kwargs)


  OK in 8.37s tools: get_context_overview, detect_temporal_columns, get_temporal_extent, detect_spatial_columns, get_spatial_extent -> acontia_cretata_react.json

[3/100] acrothamnus_colensoi.csv


...

  OK in 7.921s tools: get_context_overview, detect_temporal_columns, get_temporal_extent, detect_spatial_columns, get_spatial_extent -> wisteriopsis_reticulata_react.json

MODEL COMPLETE: 100 success, 0 skipped. Moving to the next model.

MODEL 2/2

react model: Sehyo/Qwen3.5-122B-A10B-NVFP4
Standard: croissant_inaturalist_standard
Output: /home/com3dian/Github/metadata_agent/outputs/react_inaturalist_Sehyo_Qwen3.5-122B-A10B-NVFP4
Resuming with 100/100 datasets remaining; 0 existing result files kept.

[1/100] acianthus_fornicatus.csv


/home/com3dian/Github/metadata_agent/src/baseline/ReAct/runner.py:301: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  outcome = describe_csv(csv_path, **kwargs)


  TIMEOUT -> acianthus_fornicatus_react.skipped.json

[2/100] acontia_cretata.csv


/home/com3dian/Github/metadata_agent/src/baseline/ReAct/runner.py:301: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  outcome = describe_csv(csv_path, **kwargs)


KeyboardInterrupt: 